# Bank transaction parser: QLoRA fine-tune on a free Kaggle T4

Runs the whole experiment end to end: regex baseline, base-model few-shot baseline, QLoRA fine-tune of Qwen2.5-1.5B-Instruct, evaluation on seen and unseen formats/merchants, and upload to the Hugging Face Hub.

**Settings:** Accelerator = GPU T4 x2 (one is used), Internet = on. For the upload step add a secret named `HF_TOKEN` (Add-ons → Secrets) holding a Hugging Face *write* token.

**Resuming:** if the output of an earlier run of this notebook is attached as an input, the trained adapter and base-model predictions are reused instead of being recomputed.

Code: https://github.com/Tanzil-Ali-Rizvi/bank-txn-parser-qlora

In [ ]:
import glob, os, shutil, subprocess
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)  # absent on CPU sessions
sh("rm -rf bank-txn-parser-qlora && git clone -q https://github.com/Tanzil-Ali-Rizvi/bank-txn-parser-qlora")
os.chdir("bank-txn-parser-qlora")
os.environ["PYTHONPATH"] = "src"
# Kaggle ships an old torchao that recent peft refuses to load next to; it is not needed here.
sh("pip install -q -U peft bitsandbytes && pip uninstall -q -y torchao")
sh('python -c "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)"')

prev = glob.glob("/kaggle/input/**/runs/qlora/adapter/adapter_config.json", recursive=True)
RESUME = bool(prev)
if RESUME:
    old = Path(prev[0]).parents[3]
    shutil.copytree(old / "runs" / "qlora", "runs/qlora")
    Path("results/preds").mkdir(parents=True, exist_ok=True)
    for f in glob.glob(str(old / "results/preds/base_fewshot_*.jsonl")):
        shutil.copy(f, "results/preds/")
print("Resuming from", prev[0] if RESUME else "nothing: full run")

## 1. Baselines
A hand-written regex parser (with a dictionary of every merchant seen in training) and the untuned base model with a detailed instruction plus 4 worked examples.

In [ ]:
sh("python -m txnparser.rules")
if not Path("results/preds/base_fewshot_test_ood.jsonl").exists():
    sh("python -m txnparser.infer --model Qwen/Qwen2.5-1.5B-Instruct --mode base --name base_fewshot --batch 16")
sh("python -m txnparser.report --systems rules base_fewshot")

## 2. QLoRA fine-tune
4-bit NF4 base, LoRA r=16 on all linear layers, 2 epochs over 5,000 examples, loss on the JSON answer only.

In [ ]:
if not Path("runs/qlora/adapter/adapter_config.json").exists() and not Path("results/preds/ft_test_ood.jsonl").exists():
    sh("python -m txnparser.train --data data --out runs/qlora")
    shutil.copy("runs/qlora/train_info.json", "results/train_info.json")
sh("head -40 results/train_info.json")

## 3. Evaluate the fine-tuned model
The fine-tuned model gets a one-line prompt instead of the long instruction and examples.

In [ ]:
HF_REPO = "TARizvi/bank-txn-parser-qwen2.5-1.5b"
# If the fine-tuned predictions are already committed (and the model published), reuse them: this lets the
# GGUF step below run on a CPU-only session. Delete results/preds/ft_*.jsonl to recompute from scratch.
REUSE = Path("results/preds/ft_test_ood.jsonl").exists()
if REUSE:
    from huggingface_hub import snapshot_download
    snapshot_download(HF_REPO, local_dir="runs/merged", ignore_patterns=["results/*", "adapter/*", "gguf/*", "README.md"])
else:
    sh("python -m txnparser.infer --model Qwen/Qwen2.5-1.5B-Instruct --adapter runs/qlora/adapter --mode ft --name ft --batch 32 --save-merged runs/merged")
sh("python -m txnparser.report")

## 4. Publish to the Hugging Face Hub

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi
user = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
print("Uploading as", user)
if not REUSE:
    sh(f"python -m txnparser.publish_hf --user {user} --merged runs/merged --adapter runs/qlora/adapter")

## 5. Export an 8-bit GGUF for CPU inference and check it still scores the same
The live demo runs on a free CPU host with llama.cpp. Q8_0 is checked against the fp16 model on 100 + 100 test examples.

In [ ]:
# The prebuilt CPU wheels are linked against musl libc and fail on Ubuntu, so compile from source (~5 min).
sh('CMAKE_ARGS="-DGGML_NATIVE=ON" pip install -q llama-cpp-python==0.3.19 --no-binary llama-cpp-python')
sh("rm -rf /kaggle/tmp/llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/tmp/llama.cpp")
sh("pip install -q gguf sentencepiece protobuf")
GGUF = "gguf/bank-txn-parser-qwen2.5-1.5b-q8_0.gguf"
sh(f"mkdir -p gguf && python /kaggle/tmp/llama.cpp/convert_hf_to_gguf.py runs/merged --outtype q8_0 --outfile {GGUF}")
sh(f"python -m txnparser.gguf_eval --gguf {GGUF} --n 100 && cat results/gguf_check.json")
api = HfApi(token=os.environ["HF_TOKEN"])
repo = f"{user}/bank-txn-parser-qwen2.5-1.5b"
api.upload_file(path_or_fileobj=GGUF, path_in_repo=GGUF, repo_id=repo, commit_message="Add Q8_0 GGUF for CPU inference")
api.upload_folder(folder_path="results", repo_id=repo, path_in_repo="results", allow_patterns=["gguf_check.json", "preds/gguf_*.jsonl"], commit_message="GGUF quality check")

In [ ]:
sh("mkdir -p /kaggle/working/results && cp -r results/* /kaggle/working/results/ && ls /kaggle/working/results")
# keep the notebook output small: the merged model and GGUF live on the Hub
sh("rm -rf runs/merged gguf")